# f-string formatting: alignment, width and the format spec

You'll often see a line like this one for printing a table header:

```python
print(f"{'format':<12}{'bits':>6}{'size (GB)':>12}{'vs fp32':>10}")
```

**What do `<12` and `>6` mean?**

Inside `{ }`, anything after the **colon `:`** is a *format spec*: it tells Python **how** to show the value.

| Spec | Meaning |
|------|---------|
| `<12` | Use **at least 12 characters** and **left-align** the text (extra space goes on the right) |
| `>6`  | Use **at least 6 characters** and **right-align** the text (extra space goes on the left) |

```
{  value  :  <   12 }
   ▲         ▲   ▲
   │         │   └── width: total number of characters to fill
   │         └────── alignment: < left, > right, ^ centre
   └──────────────── what to print
```

## 1. Breaking down the example

Let's print it, with `|` added at both ends so you can see where the spaces are.

In [2]:
header = f"{'format':<12}{'bits':>6}{'size (GB)':>12}{'vs fp32':>10}"
print("|" + header + "|")
print("|" + "123456789012" + "123456" + "123456789012" + "1234567890" + "|")   # ruler: 12 + 6 + 12 + 10
print(len(header))   # 12 + 6 + 12 + 10 = 40

|format        bits   size (GB)   vs fp32|
|1234567890121234561234567890121234567890|
40


Piece by piece:

| Piece | Text | Length | Width | Padding | Result |
|-------|------|--------|-------|---------|--------|
| `{'format':<12}` | `format` | 6 | 12 | 6 spaces on the **right** | `format      ` |
| `{'bits':>6}` | `bits` | 4 | 6 | 2 spaces on the **left** | `  bits` |
| `{'size (GB)':>12}` | `size (GB)` | 9 | 12 | 3 spaces on the **left** | `   size (GB)` |
| `{'vs fp32':>10}` | `vs fp32` | 7 | 10 | 3 spaces on the **left** | `   vs fp32` |

**Why do it this way?** Each column always has the same width, so the header lines up with the data rows printed below it using the same widths.

## 2. The three alignment characters

| Symbol | Alignment | Where the padding goes |
|--------|-----------|------------------------|
| `<` | left | right side |
| `>` | right | left side |
| `^` | centre | split between both sides (the extra one goes on the right) |

In [3]:
word = "cat"
print(f"|{word:<10}|   left")
print(f"|{word:>10}|   right")
print(f"|{word:^10}|   centre")

|cat       |   left
|       cat|   right
|   cat    |   centre


## 3. Default alignment (when you give only a width)

If you leave out `<`, `>` or `^`:
- **strings** are **left**-aligned
- **numbers** are **right**-aligned

That's why numeric columns usually line up neatly without extra work.

In [4]:
print(f"|{'abc':10}|")   # string → left by default
print(f"|{42:10}|")      # number → right by default
print(f"|{3.14:10}|")

|abc       |
|        42|
|      3.14|


## 4. Width is a **minimum**, not a maximum

If the value is longer than the width, Python **does not cut it**. The column just gets wider and the rest of the row shifts across.

In [5]:
print(f"|{'short':<8}|")
print(f"|{'a much longer text':<8}|")   # not truncated, so the table loses its alignment

|short   |
|a much longer text|


To **cut** a string to a maximum length, use a precision: `.N` on a string keeps only the first N characters.

In [6]:
text = "a much longer text"
print(f"|{text:<8.8}|")   # width 8, max 8 characters
print(f"|{text:.5}|")     # first 5 characters

|a much l|
|a muc|


## 5. Fill character: pad with something other than spaces

Put a single character **before** the alignment symbol: `{value:fill align width}`.

In [ ]:
print(f"{'Title':*^20}")
print(f"{'Loading':.<20}OK")
print(f"{7:0>3}")          # 007
print(f"{'':-<40}")        # a 40-character divider line

## 6. Formatting numbers inside a column

You can combine alignment and width with number formatting:

| Spec | Meaning | Example | Output |
|------|---------|---------|--------|
| `.2f` | 2 decimal places | `f"{3.14159:.2f}"` | `3.14` |
| `>8.2f` | right-aligned, width 8, 2 decimals | `f"{3.14159:>8.2f}"` | `    3.14` |
| `,` | thousands separator | `f"{1234567:,}"` | `1,234,567` |
| `_` | underscore separator | `f"{1234567:_}"` | `1_234_567` |
| `.1%` | percentage | `f"{0.256:.1%}"` | `25.6%` |
| `+` | always show the sign | `f"{5:+}"` | `+5` |
| `08.3f` | zero-pad to width 8 | `f"{3.14159:08.3f}"` | `0003.142` |
| `.2e` | scientific notation | `f"{12345.678:.2e}"` | `1.23e+04` |

In [ ]:
x = 3.14159
print(f"|{x:.2f}|")
print(f"|{x:>8.2f}|")
print(f"|{x:<8.2f}|")
print(f"|{1234567:,}|")
print(f"|{1234567:>12,}|")
print(f"|{0.256:.1%}|")
print(f"|{5:+}|  |{-5:+}|")
print(f"|{x:08.3f}|")
print(f"|{12345.678:.2e}|")

### The full format-spec layout (for reference)

```
[[fill]align][sign][#][0][width][,|_][.precision][type]
```

You rarely need all of it. The common ones are **align + width** (`<12`), **precision + type** (`.2f`), or both together (`>10.2f`).

## 7. Putting it together: a complete table

This is the kind of table the original header line belongs to. It compares the memory needed to store a **7-billion-parameter model** at different number precisions.

Formula: `size in bytes = parameters × bits / 8`

In [ ]:
params = 7_000_000_000
formats = [("fp32", 32), ("fp16", 16), ("bf16", 16), ("int8", 8), ("int4", 4)]

print(f"{'format':<12}{'bits':>6}{'size (GB)':>12}{'vs fp32':>10}")
print("-" * 40)
for name, bits in formats:
    size_gb = params * bits / 8 / 1e9
    ratio = bits / 32
    print(f"{name:<12}{bits:>6}{size_gb:>12.1f}{ratio:>9.2f}x")

Things to notice:
- The header and rows use the **same widths** (12, 6, 12, 10), so the columns line up.
- Text columns are **left**-aligned (`<`) and number columns are **right**-aligned (`>`). Right-aligned numbers keep their decimal points in line.
- In the last column, `>9.2f` plus the `x` character adds up to 10, the same as the header's `>10`.

## 8. Dynamic width: getting the width from a variable

The width can come from another expression by nesting `{ }` inside the spec.

In [ ]:
names = ["apple", "banana", "kiwi", "watermelon"]
w = max(len(n) for n in names)   # width of the longest name

for n in names:
    print(f"|{n:<{w}}|{len(n):>3}|")

## 9. Other ways to do the same thing

The same format spec works in `format()` and `str.format()`. Strings also have helper methods for this.

In [ ]:
s = "cat"
print(f"|{s:>8}|")             # f-string
print("|{:>8}|".format(s))     # str.format
print("|" + format(s, ">8") + "|")   # built-in format()

print("|" + s.ljust(8) + "|")  # same as <8
print("|" + s.rjust(8) + "|")  # same as >8
print("|" + s.center(8) + "|") # same as ^8
print("|" + s.rjust(8, "-") + "|")  # with a fill character

## 10. Common mistakes

In [ ]:
# 1) Leaving out the colon: {x<10} is a comparison (x < 10), not formatting
x = 5
print(f"{x<10}")      # True   ← comparison!
print(f"{x:<10}|")    # 5         |  ← formatting

# 2) Using a float spec on a string raises an error
try:
    print(f"{'abc':.2f}")
except ValueError as e:
    print("ValueError:", e)

# 3) Header and row widths don't match, so the columns come out crooked
print(f"{'name':<10}{'score':>6}")
print(f"{'Anant':<8}{95:>6}")   # 8 ≠ 10

## 11. Practice

Try these before you look at the solutions.

1. Print `"Python"` centred in 20 characters, padded with `=`.
2. Print `pi = 3.14159265` right-aligned in width 10 with 3 decimal places.
3. Print the table below so that the columns line up, with names left-aligned and prices right-aligned to 2 decimals:
   ```python
   items = [("Tea", 1.5), ("Coffee", 2.75), ("Sandwich", 6.0)]
   ```
4. Print `1234567.891` with a thousands separator and 2 decimals.

In [ ]:
# Your answers here

In [ ]:
# Solutions
print(f"{'Python':=^20}")

pi = 3.14159265
print(f"|{pi:>10.3f}|")

items = [("Tea", 1.5), ("Coffee", 2.75), ("Sandwich", 6.0)]
print(f"{'item':<10}{'price':>8}")
print("-" * 18)
for name, price in items:
    print(f"{name:<10}{price:>8.2f}")

print(f"{1234567.891:,.2f}")

## Cheat sheet

| Spec | Meaning |
|------|---------|
| `{x:<10}` | left-align, width 10 |
| `{x:>10}` | right-align, width 10 |
| `{x:^10}` | centre, width 10 |
| `{x:*^10}` | centre, pad with `*` |
| `{x:10}` | width 10, default alignment (strings left, numbers right) |
| `{x:.2f}` | 2 decimal places |
| `{x:>10.2f}` | right-align, width 10, 2 decimals |
| `{x:,}` | thousands separator |
| `{x:.1%}` | percentage |
| `{x:05}` | zero-pad to width 5 |
| `{s:.5}` | first 5 characters of a string |
| `{x:<{w}}` | width taken from variable `w` |

**Rule to remember:** everything after `:` describes *how* to display the value. `<`/`>`/`^` is the side the text sits on, and the number is the minimum total width.